In [21]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)
from pyspark.sql.functions import col, from_json, to_json

In [22]:
spark = (
    SparkSession.builder
    .appName("bronze-cdc")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.postgresql:postgresql:42.7.3",
            "org.apache.hadoop:hadoop-aws:3.3.4",
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")

    # IMPORTANT: Iceberg AWS/MinIO settings
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.catalog.demo.s3.access-key-id", "admin")
    .config("spark.sql.catalog.demo.s3.secret-access-key", "admin123")
    .config("spark.sql.catalog.demo.client.region", "us-east-1")

    # keep Hadoop settings too
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [23]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.gold")

DataFrame[]

In [24]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.customers_cdc (
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    op STRING,
    lsn BIGINT,
    event_ts_ms BIGINT,
    before_json STRING,
    after_json STRING
)
USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.drivers_cdc (
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    op STRING,
    lsn BIGINT,
    event_ts_ms BIGINT,
    before_json STRING,
    after_json STRING
)
USING iceberg
""")

DataFrame[]

In [25]:
def get_starting_offsets(bronze_table: str, topic_name: str) -> str:
    """
    Return Kafka startingOffsets JSON based on max consumed offset in Bronze table.
    Spark Kafka startingOffsets is inclusive, so we start at max_offset + 1.
    """
    try:
        df = spark.sql(f"""
            SELECT kafka_partition, MAX(kafka_offset) AS max_offset
            FROM {bronze_table}
            GROUP BY kafka_partition
        """)
        rows = df.collect()
        if not rows:
            return "earliest"
        offsets = {topic_name: {str(r["kafka_partition"]): int(r["max_offset"]) + 1 for r in rows}}
        return json.dumps(offsets)
    except Exception:
        return "earliest"

customers_offsets = get_starting_offsets("demo.bronze.customers_cdc", "dbserver1.public.customers")
drivers_offsets   = get_starting_offsets("demo.bronze.drivers_cdc", "dbserver1.public.drivers")

print("customers startingOffsets =", customers_offsets)
print("drivers   startingOffsets =", drivers_offsets)

customers startingOffsets = earliest
drivers   startingOffsets = earliest


In [26]:
def read_kafka_batch(topic_name: str, starting_offsets: str):
    return (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", topic_name)
        .option("startingOffsets", starting_offsets)
        .option("endingOffsets", "latest")
        .load()
        .select(
            col("topic"),
            col("partition").alias("kafka_partition"),
            col("offset").alias("kafka_offset"),
            col("timestamp").alias("kafka_timestamp"),
            col("value").cast("string").alias("value")
        )
    )

customers_raw = read_kafka_batch("dbserver1.public.customers", customers_offsets)
drivers_raw   = read_kafka_batch("dbserver1.public.drivers", drivers_offsets)

print("customers raw:", customers_raw.count())
print("drivers raw:", drivers_raw.count())

customers raw: 981
drivers raw: 537


In [27]:
customers_raw = customers_raw.filter(col("value").isNotNull())
drivers_raw   = drivers_raw.filter(col("value").isNotNull())

In [28]:
payload_schema = StructType([
    StructField("before", StructType([]), True),
    StructField("after", StructType([]), True),
    StructField("source", StructType([
        StructField("lsn", LongType(), True),
    ]), True),
    StructField("op", StringType(), True),
    StructField("ts_ms", LongType(), True),
])

envelope_schema = StructType([
    StructField("payload", payload_schema, True)
])

def parse_debezium(df):
    parsed = df.withColumn("json_data", from_json(col("value"), envelope_schema))
    return parsed.select(
        col("topic"),
        col("kafka_partition"),
        col("kafka_offset"),
        col("kafka_timestamp"),
        col("json_data.payload.op").alias("op"),
        col("json_data.payload.source.lsn").alias("lsn"),
        col("json_data.payload.ts_ms").alias("event_ts_ms"),
        to_json(col("json_data.payload.before")).alias("before_json"),
        to_json(col("json_data.payload.after")).alias("after_json"),
    )

customers_bronze = parse_debezium(customers_raw)
drivers_bronze   = parse_debezium(drivers_raw)

In [29]:
customers_bronze.orderBy("kafka_offset").show(20, truncate=False)
drivers_bronze.orderBy("kafka_offset").show(20, truncate=False)

+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----------+----------+
|topic                     |kafka_partition|kafka_offset|kafka_timestamp        |op |lsn     |event_ts_ms  |before_json|after_json|
+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----------+----------+
|dbserver1.public.customers|0              |0           |2026-04-19 14:08:36.989|r  |27752320|1776607716552|NULL       |{}        |
|dbserver1.public.customers|0              |1           |2026-04-19 14:08:36.989|r  |27752320|1776607716554|NULL       |{}        |
|dbserver1.public.customers|0              |2           |2026-04-19 14:08:36.989|r  |27752320|1776607716554|NULL       |{}        |
|dbserver1.public.customers|0              |3           |2026-04-19 14:08:36.99 |r  |27752320|1776607716555|NULL       |{}        |
|dbserver1.public.customers|0              |4           |2026-04-19 14:08:36

In [30]:
customers_bronze.writeTo("demo.bronze.customers_cdc").append()
drivers_bronze.writeTo("demo.bronze.drivers_cdc").append()

In [31]:
spark.sql("""
SELECT op, COUNT(*) AS n
FROM demo.bronze.customers_cdc
GROUP BY op
ORDER BY op
""").show()

spark.sql("""
SELECT op, COUNT(*) AS n
FROM demo.bronze.drivers_cdc
GROUP BY op
ORDER BY op
""").show()

spark.sql("""
SELECT *
FROM demo.bronze.customers_cdc
ORDER BY kafka_offset DESC
LIMIT 10
""").show(truncate=False)

+---+---+
| op|  n|
+---+---+
|  c|272|
|  d|101|
|  r|241|
|  u|270|
+---+---+

+---+---+
| op|  n|
+---+---+
|  c|111|
|  d| 74|
|  r| 76|
|  u|204|
+---+---+

+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----------+----------+
|topic                     |kafka_partition|kafka_offset|kafka_timestamp        |op |lsn     |event_ts_ms  |before_json|after_json|
+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----------+----------+
|dbserver1.public.customers|0              |984         |2026-04-19 14:26:24.131|c  |28395176|1776608783794|NULL       |{}        |
|dbserver1.public.customers|0              |983         |2026-04-19 14:26:23.127|u  |28395000|1776608782772|NULL       |{}        |
|dbserver1.public.customers|0              |982         |2026-04-19 14:26:22.121|u  |28394824|1776608781749|NULL       |{}        |
|dbserver1.public.customers|0              |98